In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# Fully connected network

In [23]:
class FCN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.gelu = nn.GELU()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.gelu(x)
        x = self.fc2(x)
        x = self.gelu(x)
        x = self.fc3(x)
        return x

In [9]:
input_size = 10
hidden_size = 4
output_size = 5

In [10]:
fcn = FCN(input_size, hidden_size, output_size)

In [11]:
batch_size = 20

In [13]:
X = torch.randn(batch_size, input_size)

In [14]:
X.shape

torch.Size([20, 10])

In [18]:
# inference
with torch.no_grad():
    y_pred = fcn(X)

In [19]:
y_pred.shape

torch.Size([20, 5])

In [20]:
y_pred.requires_grad

False

In [21]:
# Train FCN classfication

In [22]:
input_size = 7
hidden_size = 2
output_size = 10

In [24]:
fcn2 = FCN(input_size, hidden_size, output_size)

In [25]:
learning_rate = 1e-3
batch_size = 32
num_epoch = 100

In [27]:
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(fcn2.parameters(), learning_rate)

In [37]:
X_train = torch.randn(500, input_size)
y_train = torch.randint(0, output_size, (500,))
X_test = torch.randn(25, input_size)
y_test = torch.randint(0, output_size, (25,))

In [30]:
dataset = torch.utils.data.TensorDataset(X_train, y_train)
data_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last= True)

In [36]:
for epoch in range(num_epoch):
    running_loss = 0.0
    for inputs, label in data_loader:
        optimizer.zero_grad()
        loss = loss_func(fcn2(inputs), label)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if (epoch+1) % 10 == 0:
        print(f'Epoch[{epoch+1}/{num_epoch}], Loss: {running_loss/len(data_loader):.4f}')
        

Epoch[10/100], Loss: 2.3244
Epoch[20/100], Loss: 2.3141
Epoch[30/100], Loss: 2.2998
Epoch[40/100], Loss: 2.2890
Epoch[50/100], Loss: 2.2831
Epoch[60/100], Loss: 2.2759
Epoch[70/100], Loss: 2.2697
Epoch[80/100], Loss: 2.2659
Epoch[90/100], Loss: 2.2553
Epoch[100/100], Loss: 2.2576


In [ ]:
# Inference

In [49]:
with torch.no_grad():
    y_logits = fcn2(X_test)

In [50]:
y_logits.requires_grad

False

In [51]:
y_pred = torch.argmax(y_logits, axis=1)

In [52]:
y_pred.shape

torch.Size([25])

In [53]:
y_test

tensor([6, 3, 9, 8, 7, 5, 3, 4, 8, 0, 7, 3, 1, 3, 7, 0, 6, 5, 4, 2, 6, 9, 8, 7,
        7])

In [54]:
y_pred

tensor([6, 6, 2, 0, 6, 6, 0, 1, 2, 2, 2, 1, 6, 0, 6, 9, 2, 1, 6, 1, 2, 2, 0, 2,
        2])

In [57]:
#Loss func for 
# loss_func = torch.nn.BCEWithLogitsLoss()

In [58]:
# train val test split

In [59]:
X = torch.randn(500, input_size)
y = torch.randint(0, output_size, (500,))

In [60]:
from torch.utils.data import random_split

In [61]:
dataset = torch.utils.data.TensorDataset(X, y)

In [62]:
train_split = 0.7
val_split = 0.15
test_split = 0.15

In [67]:
train_size = int(len(dataset) * train_split)
val_size = int(len(dataset) * val_split)
test_size = int(len(dataset) - train_size - val_size)

In [68]:
val_size

75

In [69]:
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

In [80]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [82]:
len(train_loader)

10

In [71]:
# or you can use sklearn

In [73]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [75]:
X_train.shape

torch.Size([400, 7])

In [84]:
loss_func = torch.nn.CrossEntropyLoss()

In [85]:
optimizer = torch.optim.Adam(fcn2.parameters(), lr = learning_rate)

In [78]:
# implement training loop with eval on val dataset

In [86]:
for epoch in range(num_epoch):
    train_loss = 0.0
    for inputs, label in train_loader:
        optimizer.zero_grad()
        loss = loss_func(fcn2(inputs), label)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    with torch.no_grad():
        val_loss = 0.0
        for inputs, label in val_loader:
            pred = fcn2(inputs)
            loss = loss_func(pred, label)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    if (epoch+1) % 10 == 0:
        print(f'Epoch[{epoch+1}/{num_epoch}], training loss: {train_loss:.4f}, val loss: {val_loss:.4f}')
    

Epoch[10/100], training loss: 2.3171, val loss: 2.3205
Epoch[20/100], training loss: 2.2893, val loss: 2.3527
Epoch[30/100], training loss: 2.2867, val loss: 2.3201
Epoch[40/100], training loss: 2.2814, val loss: 2.3342
Epoch[50/100], training loss: 2.2691, val loss: 2.3081
Epoch[60/100], training loss: 2.2686, val loss: 2.3082
Epoch[70/100], training loss: 2.2663, val loss: 2.2983
Epoch[80/100], training loss: 2.2644, val loss: 2.3107
Epoch[90/100], training loss: 2.2606, val loss: 2.3262
Epoch[100/100], training loss: 2.2534, val loss: 2.3213
